# Collection Operations Report - 数据抽取

**版本**: v2.1  
**更新说明**: 添加 `natural_month_repay` 月度累计回款数据，用于月时序回收进度  
**日期**: 请根据需要修改 `dt_start` 和 `dt_end`

In [424]:
# 参数配置
dt_start = '2026-03-01'  # 开始日期
dt_end = '2026-04-30'    # 结束日期

In [425]:
from get_write_data_from_dataworks import run_sql
import pandas as pd

---

## 1. TL Data - 每日组维度指标

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: TL View 的日指标

In [426]:
sql=f'''
-- TL Data: 每日组维度（Group × Day）
-- 用于 TL View 的日指标
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  dt,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  COUNT(DISTINCT owner_name) AS ownercount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  SUM(own_uncomm_case_cnt) AS total_uncomm_case,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect,
  -- 接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0  -- 仅内催
  AND owner_bucket IN ('S0','S1','S2','M1')
GROUP BY
  owner_bucket,
  owner_group,
  dt
'''
tl_data=run_sql(sql)
tl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021505822gcqu8jmy9b91&token=KytYREZtOWVhTlYrV1kyUSt0VXNCRkw4eHRNPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkwNSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTA1ODIyZ2NxdThqbXk5YjkxIl19XSwiVmVyc2lvbiI6IjEifQ==


,module,group_id,dt,headcount,ownercount,total_owing_case,total_comm_times,total_uncomm_case,total_calls,total_connect,connect_rate
0,M1,M1-Large A,2026-03-02,11,12,1741,15986,18,16669,497,0.029816
1,M1,M1-Large A,2026-03-03,12,12,1789,12464,125,12893,388,0.030094
2,M1,M1-Large A,2026-03-04,12,12,1787,14996,50,15504,480,0.030960
3,M1,M1-Large A,2026-03-05,10,12,1759,15846,47,16422,499,0.030386
4,M1,M1-Large A,2026-03-06,12,12,1702,12850,62,13361,399,0.029863


---

## 2. STL Data - 每周模块维度指标

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: STL View 的周指标

In [427]:
sql=f'''
-- STL Data: 每周模块维度（Module × Week）
-- 用于 STL View 的周指标
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
      -- 周区间：周起始为周六
      CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1')
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
)
'''
stl_data=run_sql(sql)
stl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021507776gfru8jmy9b91&token=aEhFcVN2RnowUHJJdUlKU0xxOGFsMnlYNWZNPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkwNyx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTA3Nzc2Z2ZydThqbXk5YjkxIl19XSwiVmVyc2lvbiI6IjEifQ==


,module,week,headcount,total_owing_case,total_comm_times,total_calls,total_connect
0,M1,2026-02-28-2026-03-06,31,27351,193117,196323,5501
1,M1,2026-03-07-2026-03-13,31,32912,254298,258822,7171
2,M1,2026-03-14-2026-03-20,31,35620,236139,239692,7038
3,M1,2026-03-21-2026-03-27,30,35412,228289,231925,6573
4,M1,2026-03-28-2026-04-03,33,28811,213727,217017,6043


---

## 3. Agent Performance - Agent日明细

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: TL View drill-down

In [428]:
sql=f'''
-- Agent Performance: Agent日明细（Group × Agent × Day）
-- 用于 TL View  drill-down
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  owner_name AS agent_id,
  dt,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS total_owing_case,
  -- 覆盖
  sum(comm_times)/(sum(owing_case_cnt) - sum(own_uncomm_case_cnt)) cover_times,
  -- 拨打
  SUM(call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_times,
  SUM(art_call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as art_call_times,
  SUM(call_connect_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_connect_times,
  -- 接通时长
  SUM(call_billsec)/60/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as call_billmin,
  -- 单通时长（分钟）
  (SUM(call_billsec / 60) / (
    GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
    * COUNT(DISTINCT dt)
  )) / (
    SUM(call_connect_times) / (
      GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
      * COUNT(DISTINCT dt)
    )
  ) AS single_call_duration,
  -- 电话接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM (select * from phl_anls.tmp_liujun_ana_11_agent_process_daily WHERE dt between '{dt_start}'
  AND  '{dt_end}') a
    
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1')
  and call_8h_flag > 0
group by   owner_bucket,
  owner_group,
  owner_name,
  dt
ORDER BY
  owner_bucket, owner_group, owner_name, dt
'''
agent_performance=run_sql(sql)
agent_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021509570gstbcszipqt5&token=R09rdXhYWkxRQzJHNHk4M0xQQXJVdzhuS0lJPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkwOSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTA5NTcwZ3N0YmNzemlwcXQ1Il19XSwiVmVyc2lvbiI6IjEifQ==


,module,group_id,agent_id,dt,headcount,total_owing_case,cover_times,call_times,art_call_times,call_connect_times,call_billmin,single_call_duration,connect_rate
0,M1,M1-Large A,Ramos02,2026-03-02,1,146.0,9.198630,1343.0,475.0,37.0,72.250000,1.952703,0.027550
1,M1,M1-Large A,Ramos02,2026-03-03,1,150.0,6.885135,1019.0,322.0,30.0,16.866667,0.562222,0.029441
2,M1,M1-Large A,Ramos02,2026-03-04,1,151.0,8.893333,1335.0,471.0,45.0,26.250000,0.583333,0.033708
3,M1,M1-Large A,Ramos02,2026-03-05,1,148.0,10.689189,1582.0,560.0,43.0,23.683333,0.550775,0.027181
4,M1,M1-Large A,Ramos02,2026-03-06,1,142.0,8.584507,1220.0,455.0,34.0,21.950000,0.645588,0.027869


---

## 4. Group Performance - 组周明细

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: STL View drill-down

In [429]:
sql=f'''
-- Group Performance: 组周明细（Module × Group × Week）
-- 用于 STL View drill-down
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
  owner_group AS group_id,
  CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS total_owing_case,
  -- 覆盖
  sum(comm_times)/(sum(owing_case_cnt) - sum(own_uncomm_case_cnt)) cover_times,
  -- 拨打
  SUM(call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_times,
  SUM(art_call_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as art_call_times,
  SUM(call_connect_times)/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) AS call_connect_times,
  -- 接通时长
  SUM(call_billsec)/60/COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END)/count(distinct dt) as call_billmin,
  -- 单通时长（分钟）
  (SUM(call_billsec / 60) / (
    GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
    * COUNT(DISTINCT dt)
  )) / (
    SUM(call_connect_times) / (
      GREATEST(COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END), 1)
      * COUNT(DISTINCT dt)
    )
  ) AS single_call_duration,
  -- 电话接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1')
  and call_8h_flag=1
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  owner_group,
  CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) 
'''
group_performance=run_sql(sql)
group_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021512739gnx8fsz5fwz1&token=MWF6by9uY1VJMnk5cTdmaU05MEl5YStOaWRZPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkxMix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTEyNzM5Z254OGZzejVmd3oxIl19XSwiVmVyc2lvbiI6IjEifQ==


,module,group_id,week,headcount,total_owing_case,cover_times,call_times,art_call_times,call_connect_times,call_billmin,single_call_duration,connect_rate
0,M1,M1-Large A,2026-02-28-2026-03-06,12,138.983333,8.604611,1200.783333,490.433333,36.316667,33.716944,0.928415,0.030244
1,M1,M1-Large A,2026-03-07-2026-03-13,12,132.041667,9.980070,1353.138889,557.097222,38.930556,31.802778,0.816910,0.028771
2,M1,M1-Large A,2026-03-14-2026-03-20,12,155.277778,8.499538,1319.666667,535.458333,39.513889,34.250000,0.866784,0.029942
3,M1,M1-Large A,2026-03-21-2026-03-27,12,147.763889,8.368743,1218.319444,496.500000,38.486111,33.937037,0.881800,0.031590
4,M1,M1-Large A,2026-03-28-2026-04-03,14,121.485714,9.874804,1202.185714,492.942857,34.457143,30.701905,0.891017,0.028662


---

## 5. 日目标回款 + Achievement（Agent日粒度）

**源表**: `phl_anls.rpt_coll_repay_target_dly4`  
**用途**: Module Daily Trends、Risk Analysis、Achievement 计算  
**说明**: 此表已包含日粒度的目标回款数据（target_repay_principal），可直接计算 achievement。用于TL tab

In [430]:
sql=f'''
-- 日目标回款 + Achievement
-- 源表: phl_anls.rpt_coll_repay_target_dly4
-- 用途: Module Daily Trends、Risk Analysis
SELECT
  dt,
  owner_bucket,
  owner_group,
  owner_name,
  SUM(owing_principal) AS daily_owing_principal,
  SUM(repay_principal) / NULLIF(SUM(owing_principal), 0) AS daily_repay_rate,
  SUM(target_repay_principal) AS target_repay_principal,
  SUM(repay_principal) AS actual_repay_principal,
  CASE WHEN SUM(target_repay_principal) <= 0 THEN 0
       ELSE SUM(repay_principal) / SUM(target_repay_principal)
  END AS achieve_rate
FROM (
  SELECT
    *,
    CASE
      WHEN owner_bucket = 'S0' AND case_overdue_days = -3 THEN 'H3'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -2 THEN 'H2'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -1 THEN 'H1'
      WHEN owner_bucket = 'S0' AND case_overdue_days >= 0 THEN 'H0'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 1 THEN '<=1'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 5 THEN '2-5'
      WHEN owner_bucket = 'S1' AND case_overdue_days >= 6 THEN '6-7'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 8 THEN '<=8'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 12 THEN '9-12'
      WHEN owner_bucket = 'S2' AND case_overdue_days >= 13 THEN '13-15'
      WHEN owner_bucket = 'M1' AND case_overdue_days <= 16 THEN '<=16'
      WHEN owner_bucket = 'M1' AND case_overdue_days >= 17 THEN '17-25'
      WHEN owner_bucket = 'M1' THEN '26-30'
      ELSE case_overdue_days
    END AS case_stage,
    CASE
      WHEN case_owing_principal <= 1000 THEN '<=1000'
      WHEN case_owing_principal <= 2000 THEN '1000-2000'
      WHEN case_owing_principal <= 5000 THEN '2000-5000'
      WHEN case_owing_principal > 5000 THEN '>5000'
    END AS principal_stage
  FROM phl_anls.rpt_coll_repay_target_dly4
  WHERE dt between '{dt_start}' and '{dt_end}'
    AND case_overdue_days > -99
    AND owner_bucket IN ('S0', 'S1', 'S2', 'M1')
) t
WHERE owner_bucket IN ('M1', 'S1', 'S2', 'S0')
  AND TO_CHAR(dt, 'yyyyMMdd') BETWEEN REPLACE('{dt_start}', '-', '') AND REPLACE('{dt_end}', '-', '')
GROUP BY
  dt,
  owner_bucket,
  owner_group,
  owner_name
HAVING SUM(owing_principal) > 0
ORDER BY dt DESC, owner_bucket DESC, owner_group ASC
'''
daily_target_agent_repay=run_sql(sql)
daily_target_agent_repay.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021514936goy8fsz5fwz1&token=aVJpRnk3SE1GcU5RViswdnJRbUxWS0tYL2tNPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkxNSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTE0OTM2Z295OGZzejVmd3oxIl19XSwiVmVyc2lvbiI6IjEifQ==


,dt,owner_bucket,owner_group,owner_name,daily_owing_principal,daily_repay_rate,target_repay_principal,actual_repay_principal,achieve_rate
0,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI59,505172,0,10297.865968589999989703,0,0
1,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI48,508599,0,33615.928569285599932766,0,0
2,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI45,542227.08,0,9400.137435874799990599,0,0
3,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI88,492332,0,21804.184787209200087216,0,0
4,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI68,559021.12,0,9752.723306792000009751,0,0


## 5.2 日目标回款 + Achievement（组周粒度）

**源表**: `phl_anls.rpt_coll_repay_target_dly4`  
**用途**: STL tab下的Module Daily Trends、Risk Analysis、Achievement 计算  
**说明**: 此表已包含日粒度的目标回款数据（target_repay_principal），可直接计算 achievement

In [431]:
sql=f'''
-- 日目标回款 + Achievement
-- 源表: phl_anls.rpt_coll_repay_target_dly4
-- 用途: STL tab下的Module Daily Trends、Risk Analysis
SELECT
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  owner_bucket,
  owner_group,
  owner_name,
  SUM(owing_principal) AS daily_owing_principal,
  SUM(repay_principal) / NULLIF(SUM(owing_principal), 0) AS daily_repay_rate,
  SUM(target_repay_principal) AS target_repay_principal,
  SUM(repay_principal) AS actual_repay_principal,
  CASE WHEN SUM(target_repay_principal) <= 0 THEN 0
       ELSE SUM(repay_principal) / SUM(target_repay_principal)
  END AS achieve_rate
FROM (
  SELECT
    *,
    CASE
      WHEN owner_bucket = 'S0' AND case_overdue_days = -3 THEN 'H3'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -2 THEN 'H2'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -1 THEN 'H1'
      WHEN owner_bucket = 'S0' AND case_overdue_days >= 0 THEN 'H0'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 1 THEN '<=1'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 5 THEN '2-5'
      WHEN owner_bucket = 'S1' AND case_overdue_days >= 6 THEN '6-7'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 8 THEN '<=8'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 12 THEN '9-12'
      WHEN owner_bucket = 'S2' AND case_overdue_days >= 13 THEN '13-15'
      WHEN owner_bucket = 'M1' AND case_overdue_days <= 16 THEN '<=16'
      WHEN owner_bucket = 'M1' AND case_overdue_days >= 17 THEN '17-25'
      WHEN owner_bucket = 'M1' THEN '26-30'
      ELSE case_overdue_days
    END AS case_stage,
    CASE
      WHEN case_owing_principal <= 1000 THEN '<=1000'
      WHEN case_owing_principal <= 2000 THEN '1000-2000'
      WHEN case_owing_principal <= 5000 THEN '2000-5000'
      WHEN case_owing_principal > 5000 THEN '>5000'
    END AS principal_stage
  FROM phl_anls.rpt_coll_repay_target_dly4
  WHERE dt between '{dt_start}' and '{dt_end}'
    AND case_overdue_days > -99
    AND owner_bucket IN ('S0', 'S1', 'S2', 'M1')
) t
WHERE owner_bucket IN ('M1', 'S1', 'S2', 'S0')
  AND TO_CHAR(dt, 'yyyyMMdd') BETWEEN REPLACE('{dt_start}', '-', '') AND REPLACE('{dt_end}', '-', '')
GROUP BY
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ),
  owner_bucket,
  owner_group,
  owner_name
HAVING SUM(owing_principal) > 0
ORDER BY week DESC, owner_bucket DESC, owner_group ASC
'''
daily_target_group_repay=run_sql(sql)
daily_target_group_repay.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021519914gjdgffi0br&token=bjVldXg0UURJUXpSa2ZwTUFEa2huaFVPaUo4PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkyMCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTE5OTE0Z2pkZ2ZmaTBiciJdfV0sIlZlcnNpb24iOiIxIn0=


,week,owner_bucket,owner_group,owner_name,daily_owing_principal,daily_repay_rate,target_repay_principal,actual_repay_principal,achieve_rate
0,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,outer_AJCAI66,2795041,0.009957993460561044,139229.968477773333547601,27833,0.19990667457806154
1,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,outer_AJCAI60,2724277.25,0.010335952407193504,87027.343626445933223658,28158,0.323553481315765752
2,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,outer_AJCAI59,2501986,0.016093615232059652,45986.728462190000026583,40266,0.875600447053032983
3,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,outer_AJCAI48,2423736,0.00963966372575231,106173.123463580133276975,23364,0.220055690534661523
4,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,outer_AJCAI45,2491321.12,0.013351951995654418,48181.940750756799949058,33264,0.690383149405983985


---

---

## 6. Natural Month Repay - 月度累计回款

**源表**: `tmp_maoruochen_phl_repay_natural_day_daily`  
**用途**: 月时序回收进度（Month Trends）  
**说明**: `dt` 为业务后一天，`substr(dt,9,2)='01'` 表示"上月月末累计状态"

In [432]:
# Natural Month Repay: 月度累计回款
# dt 为业务后一天，因此 substr(dt,9,2)='01' 表示"上月月末累计状态"
sql=f'''
SELECT
  dt_biz,
  agent_bucket,
  group_name,
  SUM(repay_principal) AS repay_principal,
  SUM(start_owing_principal) AS start_owing_principal,
  SUM(repay_principal) / SUM(start_owing_principal) as repay_rate
FROM phl_anls.tmp_maoruochen_phl_repay_natural_day_daily
WHERE data_level = '4.组别层级'  
  AND dt_biz between '{dt_start}' and '{dt_end}'
  AND group_name is not null 
GROUP BY dt_biz, agent_bucket, group_name
;
'''
natural_month_repay=run_sql(sql)
natural_month_repay.head(100)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021525161g229fsz5fwz1&token=azBZU3ErbkVwTXgxWGgxWkRJcnRyRkxzaEhFPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkyNSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTI1MTYxZzIyOWZzejVmd3oxIl19XSwiVmVyc2lvbiI6IjEifQ==


,dt_biz,agent_bucket,group_name,repay_principal,start_owing_principal,repay_rate
0,2026-03-01,M1,Target,0.00858407,1,0.00858407
1,2026-03-02,M1,Target,0.01131324,1,0.01131324
2,2026-03-03,M1,Target,0.01626744,1,0.01626744
3,2026-03-04,M1,Target,0.02060566,1,0.02060566
4,2026-03-05,M1,Target,0.02498771,1,0.02498771
...,...,...,...,...,...,...
95,2026-03-03,M1_Small,Target,0.02321835,1,0.02321835
96,2026-03-04,M1_Small,Target,0.03013967,1,0.03013967
97,2026-03-05,M1_Small,Target,0.03671168,1,0.03671168
98,2026-03-06,M1_Small,Target,0.0437456,1,0.0437456


---

## 7. PTP Group Data - 人日PTP明细

**源表**: `phl_anls.tmp_zyt_agent_ptp`  
**用途**: TL View drill-down PTP履约率  
**粒度**: `owner_name × dt`

In [433]:
sql=f'''
-- PTP 承诺还款 - 人日明细（Agent × Day）
-- 用于 TL View drill-down
SELECT
  dt,
  owner_group,
  owner_name,
  SUM(ptp_case) AS ptp_case_cnt,
  SUM(ptp_today_case) AS ptp_today_case,
  SUM(today_actual_repay_case) AS today_actual_repay_case,
  CASE WHEN SUM(ptp_today_case) > 0
       THEN SUM(today_actual_repay_case) / SUM(ptp_today_case)
       ELSE 0
  END AS today_ptp_repay_rate
FROM phl_anls.tmp_zyt_agent_ptp
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
GROUP BY dt, owner_group, owner_name
ORDER BY dt, owner_group ASC
'''
ptp_agent_data = run_sql(sql)
ptp_agent_data.head(10)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021531967g549fsz5fwz1&token=WnhWRDBVdjVvYzVUNXoxcVBZaW1yYmtrWElFPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkzMix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTMxOTY3ZzU0OWZzejVmd3oxIl19XSwiVmVyc2lvbiI6IjEifQ==


,dt,owner_group,owner_name,ptp_case_cnt,ptp_today_case,today_actual_repay_case,today_ptp_repay_rate
0,2026-03-01,None,garcia03,0,0,0,0.0
1,2026-03-01,M4+-B-AJCAI,outer_AJCAI72,0,0,0,0.0
2,2026-03-01,M4+-B-AJCAI,outer_AJCAI75,0,0,0,0.0
3,2026-03-01,M4+-B-AJCAI,outer_AJCAI74,0,0,0,0.0
4,2026-03-01,M4+-B-AJCAI,outer_AJCAI71,0,0,0,0.0
5,2026-03-01,M4+-B-AJCAI,outer_AJCAI73,0,0,0,0.0
6,2026-03-01,Cluster A-Nocall,S1-NOCA,0,0,0,0.0
7,2026-03-01,Cluster A-Nocall,S0-NOCA,0,0,0,0.0
8,2026-03-01,L0 Bucket,suderiond,0,0,0,0.0
9,2026-03-01,L0 Bucket,garjas03,0,0,0,0.0


---

## 8. PTP Group Data - 组周PTP明细

**源表**: `phl_anls.tmp_zyt_agent_ptp`  
**用途**: STL View drill-down PTP履约率  
**粒度**: `owner_group × week`

In [434]:
sql=f'''
-- PTP 承诺还款 - 组周明细（Group × Week）
-- 用于 STL View drill-down
SELECT
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  owner_group,
  SUM(ptp_case) AS ptp_case_cnt,
  SUM(ptp_today_case) AS ptp_today_case,
  SUM(today_actual_repay_case) AS today_actual_repay_case,
  CASE WHEN SUM(ptp_today_case) > 0
       THEN SUM(today_actual_repay_case) / SUM(ptp_today_case)
       ELSE 0
  END AS today_ptp_repay_rate
FROM phl_anls.tmp_zyt_agent_ptp
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
GROUP BY   CONCAT(
  -- 周起始：周六
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  ),
  '-',
  -- 周结束：周五
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  )
), owner_group
ORDER BY week DESC, owner_group ASC
'''
ptp_group_data = run_sql(sql)
ptp_group_data.head()

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021534466gl2ao631zw6&token=MXJTSkM1ZVg4ZkJwWG5QTVdnYTNBOE9JQUV3PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkzNCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTM0NDY2Z2wyYW82MzF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,week,owner_group,ptp_case_cnt,ptp_today_case,today_actual_repay_case,today_ptp_repay_rate
0,2026-04-25-2026-05-01,None,0,0,0,0.000000
1,2026-04-25-2026-05-01,M4+-B-AJCAI,3,3,0,0.000000
2,2026-04-25-2026-05-01,L0,125,87,74,0.850575
3,2026-04-25-2026-05-01,L1,130,103,64,0.621359
4,2026-04-25-2026-05-01,L2,15,7,3,0.428571


---

## 9. 呼损率 - 人日明细

**源表**: `phl_anls.tmp_phl_zyt_20_lark_genesys_daily`  
**用途**: TL View drill-down 呼损率监控  
**筛选**: `call_type = '3'`（仅一键外呼）  
**粒度**: `owner_name × dt`

In [435]:
sql=f'''
-- 呼损率数据 - 人日明细（Agent × Day）
-- 用于 STL View drill-down
-- 仅统计一键外呼 (call_type = '3')
SELECT
  dt,
  group_name,
  owner_name,
  SUM(call_loss) / (SUM(call_loss) + SUM(connect_times)) AS call_loss_rate,
  SUM(call_loss) AS call_loss,
  SUM(connect_times) AS connect_times
FROM phl_anls.tmp_phl_zyt_20_lark_genesys_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND CAST(call_type AS STRING) = '3'
GROUP BY dt, group_name, owner_name
ORDER BY dt DESC, group_name ASC
'''
call_loss_agent_data = run_sql(sql)
call_loss_agent_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021536661gn1v8jmy9b91&token=RWhUTWVBQm50MmNWUWw2WlB4eFoxOXo2Q2JJPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkzNix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTM2NjYxZ24xdjhqbXk5YjkxIl19XSwiVmVyc2lvbiI6IjEifQ==


,dt,group_name,owner_name,call_loss_rate,call_loss,connect_times
0,2026-04-28,M4+-B-AJCAI,outer_AJCAI73,0.062500,1,15
1,2026-04-28,M4+-B-AJCAI,outer_AJCAI75,0.750000,15,5
2,2026-04-28,M4+-B-AJCAI,outer_AJCAI71,NaN,0,0
3,2026-04-28,M4+-B-AJCAI,outer_AJCAI74,0.312500,5,11
4,2026-04-28,L0,marbellajp,0.142857,1,6


---

## 10. 呼损率 - 组周明细

**源表**: `phl_anls.tmp_phl_zyt_20_lark_genesys_daily`  
**用途**: STL View drill-down 呼损率监控  
**筛选**: `call_type = '3'`（仅一键外呼）  
**粒度**: `group_name × week`

In [436]:
sql=f'''
-- 呼损率数据 - 组周明细（Group × Week）
-- 用于 STL View drill-down
-- 仅统计一键外呼 (call_type = '3')
SELECT
CONCAT(
  -- 周起始：周六
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  ),
  '-',
  -- 周结束：周五
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  )
) AS week,
  group_name,
  SUM(call_loss) / (SUM(call_loss) + SUM(connect_times)) AS call_loss_rate,
  SUM(call_loss) AS call_loss,
  SUM(connect_times) AS connect_times
FROM phl_anls.tmp_phl_zyt_20_lark_genesys_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND CAST(call_type AS STRING) = '3'
GROUP BY   CONCAT(
  -- 周起始：周六
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  ),
  '-',
  -- 周结束：周五
  TO_CHAR(
    DATEADD(
      TO_DATE(dt, 'yyyy-MM-dd'), 
      ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
      'dd'
    ), 
    'yyyy-MM-dd'
  )
), group_name
ORDER BY week DESC, group_name ASC
'''
call_loss_group_data = run_sql(sql)
call_loss_group_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021539626gu4ao631zw6&token=MzRaY1Blc0RIbitBVU1JK2VBdmh4RklHeDkwPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDkzOSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTM5NjI2Z3U0YW82MzF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,week,group_name,call_loss_rate,call_loss,connect_times
0,2026-04-25-2026-05-01,M4+-B-AJCAI,0.270718,49,132
1,2026-04-25-2026-05-01,L0,0.386973,101,160
2,2026-04-25-2026-05-01,L1,0.325613,239,495
3,2026-04-25-2026-05-01,L2,0.353535,105,192
4,2026-04-25-2026-05-01,L3,0.391026,61,95


# 11. 出勤率-组 日明细

In [437]:
sql=f'''
-- 出勤率数据 - 组日（Group × Day）
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  dt,
  -- 人力
  COUNT(CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) / COUNT(owner_name)AS attd_rate_8h
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1')
--   and call_8h_flag > 0
group by   owner_bucket,
  owner_group,
--   owner_name,
  dt
ORDER BY
  owner_bucket, owner_group,dt

'''
attd_group_daily_data = run_sql(sql)
attd_group_daily_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=2026042902154345gp6ao631zw6&token=TFJXYjROcjZ2SE9lNXZFOTduMnNsTlF6cE8wPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDk0Myx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTQzNDVncDZhbzYzMXp3NiJdfV0sIlZlcnNpb24iOiIxIn0=


,module,group_id,dt,attd_rate_8h
0,M1,M1-Large A,2026-03-02,0.875000
1,M1,M1-Large A,2026-03-03,1.000000
2,M1,M1-Large A,2026-03-04,1.000000
3,M1,M1-Large A,2026-03-05,0.871795
4,M1,M1-Large A,2026-03-06,1.000000


# 12. 出勤率-组 周明细

In [438]:
sql=f'''
-- 出勤率数据 - 组周（Group × Week）
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  -- 人力
  COUNT(CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) / COUNT(owner_name)AS attd_rate_8h
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1')
group by   owner_bucket,
  owner_group,
  CONCAT(
  -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) 
ORDER BY
  owner_bucket, owner_group,week

'''
attd_group_week_data = run_sql(sql)
attd_group_week_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021545947grngffi0br&token=WVZ3N0F3ZXBVUkY4WWNEK1VWdXVMYWI4aHJNPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDk0Nix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTQ1OTQ3Z3JuZ2ZmaTBiciJdfV0sIlZlcnNpb24iOiIxIn0=


,module,group_id,week,attd_rate_8h
0,M1,M1-Large A,2026-02-28-2026-03-06,0.952880
1,M1,M1-Large A,2026-03-07-2026-03-13,0.972222
2,M1,M1-Large A,2026-03-14-2026-03-20,1.000000
3,M1,M1-Large A,2026-03-21-2026-03-27,0.948718
4,M1,M1-Large A,2026-03-28-2026-04-03,1.000000


# 13. by逾期天数和案件金额-人 日

In [439]:
sql=f'''
-- 源表: phl_anls.rpt_coll_repay_target_dly4
-- 用途: Drill Down、Risk Analysis
SELECT
  dt,
  owner_bucket,
  owner_group,
  owner_name,
  case_stage,
  principal_stage,
  SUM(owing_principal) AS owing_principal,
  sum(repay_principal) as repay_principal
FROM (
  SELECT
    *,
    CASE
      WHEN owner_bucket = 'S0' AND case_overdue_days = -3 THEN 'H3'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -2 THEN 'H2'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -1 THEN 'H1'
      WHEN owner_bucket = 'S0' AND case_overdue_days >= 0 THEN 'H0'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 1 THEN '<=1'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 5 THEN '2-5'
      WHEN owner_bucket = 'S1' AND case_overdue_days >= 6 THEN '6-7'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 8 THEN '<=8'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 12 THEN '9-12'
      WHEN owner_bucket = 'S2' AND case_overdue_days >= 13 THEN '13-15'
      WHEN owner_bucket = 'M1' AND case_overdue_days <= 16 THEN '<=16'
      WHEN owner_bucket = 'M1' AND case_overdue_days >= 17 THEN '17-25'
      WHEN owner_bucket = 'M1' THEN '26-30'
      ELSE case_overdue_days
    END AS case_stage,
    CASE
      WHEN case_owing_principal <= 1000 THEN '<=1000'
      WHEN case_owing_principal <= 2000 THEN '1000-2000'
      WHEN case_owing_principal <= 5000 THEN '2000-5000'
      WHEN case_owing_principal > 5000 THEN '>5000'
    END AS principal_stage
  FROM phl_anls.rpt_coll_repay_target_dly4
  WHERE dt between '{dt_start}' and '{dt_end}'
    AND case_overdue_days > -99
    AND owner_bucket IN ('S0', 'S1', 'S2', 'M1')
) t
WHERE owner_bucket IN ('M1', 'S1', 'S2', 'S0')
  AND TO_CHAR(dt, 'yyyyMMdd') BETWEEN REPLACE('{dt_start}', '-', '') AND REPLACE('{dt_end}', '-', '')
GROUP BY
  dt,
  owner_bucket,
  owner_group,
  owner_name,
  case_stage,
  principal_stage
HAVING SUM(owing_principal) > 0
ORDER BY dt DESC, owner_bucket DESC, owner_group ASC
'''
daily_target_agent_breakdown=run_sql(sql)
daily_target_agent_breakdown.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021548267gy8ccszipqt5&token=S2ZZZVp4VWxxVDVRYm5uTUNqRnpjcHZXY25ZPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDk0OSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTQ4MjY3Z3k4Y2NzemlwcXQ1Il19XSwiVmVyc2lvbiI6IjEifQ==


,dt,owner_bucket,owner_group,owner_name,case_stage,principal_stage,owing_principal,repay_principal
0,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI68,<=8,2000-5000,28059,0
1,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI68,<=8,1000-2000,5175,0
2,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI68,9-12,>5000,183650,0
3,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI68,9-12,2000-5000,69887.12,0
4,2026-04-29,S2,S2-AJCAI-LARGE,outer_AJCAI88,<=8,>5000,46000,0


# 14. by逾期天数和案件金额-组 周

In [440]:
sql=f'''
-- 日目标回款 + Achievement
-- 源表: phl_anls.rpt_coll_repay_target_dly4
-- 用途: STL tab下的Drill Down、Risk Analysis
SELECT
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ) AS week,
  owner_bucket,
  owner_group,
  case_stage,
  principal_stage,
  SUM(owing_principal) AS owing_principal,
  sum(repay_principal) as repay_principal
FROM (
  SELECT
    *,
    CASE
      WHEN owner_bucket = 'S0' AND case_overdue_days = -3 THEN 'H3'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -2 THEN 'H2'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -1 THEN 'H1'
      WHEN owner_bucket = 'S0' AND case_overdue_days >= 0 THEN 'H0'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 1 THEN '<=1'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 5 THEN '2-5'
      WHEN owner_bucket = 'S1' AND case_overdue_days >= 6 THEN '6-7'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 8 THEN '<=8'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 12 THEN '9-12'
      WHEN owner_bucket = 'S2' AND case_overdue_days >= 13 THEN '13-15'
      WHEN owner_bucket = 'M1' AND case_overdue_days <= 16 THEN '<=16'
      WHEN owner_bucket = 'M1' AND case_overdue_days >= 17 THEN '17-25'
      WHEN owner_bucket = 'M1' THEN '26-30'
      ELSE case_overdue_days
    END AS case_stage,
    CASE
      WHEN case_owing_principal <= 1000 THEN '<=1000'
      WHEN case_owing_principal <= 2000 THEN '1000-2000'
      WHEN case_owing_principal <= 5000 THEN '2000-5000'
      WHEN case_owing_principal > 5000 THEN '>5000'
    END AS principal_stage
  FROM phl_anls.rpt_coll_repay_target_dly4
  WHERE dt between '{dt_start}' and '{dt_end}'
    AND case_overdue_days > -99
    AND owner_bucket IN ('S0', 'S1', 'S2', 'M1')
) t
WHERE owner_bucket IN ('M1', 'S1', 'S2', 'S0')
  AND TO_CHAR(dt, 'yyyyMMdd') BETWEEN REPLACE('{dt_start}', '-', '') AND REPLACE('{dt_end}', '-', '')
GROUP BY
    CONCAT(
      -- 周起始：周六
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          -((WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 2) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      ),
      '-',
      -- 周结束：周五
      TO_CHAR(
        DATEADD(
          TO_DATE(dt, 'yyyy-MM-dd'), 
          ((11 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd'))) % 7), 
          'dd'
        ), 
        'yyyy-MM-dd'
      )
    ),
  owner_bucket,
  owner_group,
  case_stage,
  principal_stage
HAVING SUM(owing_principal) > 0
ORDER BY week DESC, owner_bucket DESC, owner_group ASC
'''
week_target_group_breakdown=run_sql(sql)
week_target_group_breakdown.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021555408g2cao631zw6&token=M25XcjRFNXpqRFdSMnpweUg3K3RRK1B3ZXNNPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDk1NSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNTU1NDA4ZzJjYW82MzF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,week,owner_bucket,owner_group,case_stage,principal_stage,owing_principal,repay_principal
0,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,9-12,<=1000,984,0
1,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,9-12,1000-2000,780668.95,16702
2,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,9-12,2000-5000,2799385.09,23874
3,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,9-12,>5000,6347751.08,60551
4,2026-04-25-2026-05-01,S2,S2-AJCAI-LARGE,<=8,<=1000,1626,0


# 15. 催员架构

In [441]:
sql=f'''
    select group_name, owner_name, owner_id, to_date(start_date) as start_date
    ,to_date(end_date) end_date
    from phl_data.edw_fact_sword_coll_owner_detail_info_zipper 
    where app = 'juanhand'
    and group_name is not null
    and owner_role_name = 'Agent'
    and to_date(end_date)>'{dt_start}'
'''
agent_org=run_sql(sql)
agent_org.head(30)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260429021600877g9fccszipqt5&token=SXRRK0l0dlpuYVRrWWVrQnlFYXNRMXNyVkxZPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDk2MCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNjAwODc3ZzlmY2NzemlwcXQ1Il19XSwiVmVyc2lvbiI6IjEifQ==


,group_name,owner_name,owner_id,start_date,end_date
0,S1-Large D,jamen01,123,2025-12-01,2999-12-31
1,M2+-C-MBA,outer_MBA,1126,2024-10-10,2999-12-31
2,T2-B Bucket,privado01,1137,2025-08-01,2999-12-31
3,T4-F Bucket,aquino02,1158,2026-03-01,2999-12-31
4,S0-AI,S0-AI,1163,2025-02-27,2999-12-31
5,Cluster A-Nocall,S0-NOCA,1164,2025-02-27,2999-12-31
6,S2-AI,S2-AI,1167,2025-02-27,2999-12-31
7,Cluster B-Nocall,M1-NOCA,1170,2025-02-27,2999-12-31
8,Cluster B-Nocall,M2-NOCA,1172,2025-02-27,2999-12-31
9,S2-Large B,isip01,1177,2026-03-01,2999-12-31


# 15. 催员架构

In [442]:
sql=f'''
    select group_name, owner_name, owner_id, to_date(start_date) as start_date
    ,to_date(end_date) end_date
    from phl_data.edw_fact_sword_coll_owner_detail_info_zipper 
    where app = 'juanhand'
    and group_name is not null
    and owner_role_name = 'Agent'
    and to_date(end_date)>'{dt_start}'
'''
agent_org=run_sql(sql)
agent_org.head(30)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=2026042902160214gseao631zw6&token=MDZ4cmNNSnlkR2RwaHdwSWJMUjROckFoRDhJPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc4MDAyMDk2Mix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwNDI5MDIxNjAyMTRnc2VhbzYzMXp3NiJdfV0sIlZlcnNpb24iOiIxIn0=


,group_name,owner_name,owner_id,start_date,end_date
0,S1-Large D,jamen01,123,2025-12-01,2999-12-31
1,M2+-C-MBA,outer_MBA,1126,2024-10-10,2999-12-31
2,T2-B Bucket,privado01,1137,2025-08-01,2999-12-31
3,T4-F Bucket,aquino02,1158,2026-03-01,2999-12-31
4,S0-AI,S0-AI,1163,2025-02-27,2999-12-31
5,Cluster A-Nocall,S0-NOCA,1164,2025-02-27,2999-12-31
6,S2-AI,S2-AI,1167,2025-02-27,2999-12-31
7,Cluster B-Nocall,M1-NOCA,1170,2025-02-27,2999-12-31
8,Cluster B-Nocall,M2-NOCA,1172,2025-02-27,2999-12-31
9,S2-Large B,isip01,1177,2026-03-01,2999-12-31


## 输出说明

| Cell | 输出变量名 | 用途 | drill-down 层级 |
|------|-----------|------|----------------|
| 1 | `tl_data` | TL每日指标 | — |
| 2 | `stl_data` | STL每周指标 | — |
| 3 | `agent_performance` | Agent日明细 | TL drill-down |
| 4 | `group_performance` | 组周明细 | STL drill-down |
| 5.1 | `daily_target_agent_repay` | Agent日目标回款（含achievement） | TL |
| 5.2 | `daily_target_agent_repay` | 组周目标回款（含achievement） | STL |
| 6 | `natural_month_repay` | 月度累计回款（Month Trends） | — |
| 7 | `ptp_agent_data` | Agent日PTP履约明细 | TL drill-down |
| 8 | `ptp_group_data` | 组周PTP履约明细 | STL drill-down |
| 9 | `call_loss_data` | Agent日呼损率 | TL drill-down |
| 10 | `call_loss_data` | 组周呼损率 | STL drill-down |
| 11 | `attd_group_daily_data` | 组日出勤率（工作满8h才计入） | TL |
| 12 | `attd_group_week_data` | 组周出勤率（工作满8h才计入） | STL |
| 13 | `daily_target_agent_breakdown` | Agent日目标breakdown（by逾期阶段和案件金额） | TL |
| 14 | `week_target_group_breakdown` | 组周目标breakdown（by逾期阶段和案件金额）  | STL |
| 15 | `agent_org` | 催员组织架构  | all |

In [443]:
# 输出为 Excel（推荐）
with pd.ExcelWriter('260318_output_automation_v3.xlsx') as writer:
    tl_data.to_excel(writer, sheet_name='tl_data', index=True)
    stl_data.to_excel(writer, sheet_name='stl_data', index=True)
    agent_performance.to_excel(writer, sheet_name='agent_performance', index=True)
    group_performance.to_excel(writer, sheet_name='group_performance', index=True)
    daily_target_agent_repay.to_excel(writer, sheet_name='daily_target_agent_repay', index=True)
    daily_target_group_repay.to_excel(writer, sheet_name='daily_target_group_repay', index=True)
    natural_month_repay.to_excel(writer, sheet_name='natural_month_repay', index=True)
    ptp_agent_data.to_excel(writer, sheet_name='ptp_agent_data', index=True)
    ptp_group_data.to_excel(writer, sheet_name='ptp_group_data', index=True)
    call_loss_agent_data.to_excel(writer, sheet_name='call_loss_agent_data', index=True)
    call_loss_group_data.to_excel(writer, sheet_name='call_loss_group_data', index=True)
    attd_group_daily_data.to_excel(writer, sheet_name='attd_group_daily_data', index=True)
    attd_group_week_data.to_excel(writer, sheet_name='attd_group_week_data', index=True)
    daily_target_agent_breakdown.to_excel(writer, sheet_name='daily_target_agent_breakdown', index=True)
    week_target_group_breakdown.to_excel(writer, sheet_name='week_target_group_breakdown', index=True)
    agent_org.to_excel(writer, sheet_name='agent_org', index=True)